# 0. Imports

In [ ]:
import os
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torch.nn as nn

# 1. Configuration

In [ ]:
SR = 22050 # Sampling rate
TARGET_DURATION = 30.0 # Output duration, in seconds
N_MELS = 128 # Number of mel bands to generate
N_FFT = 2048 # FFT length
HOP_LENGTH = 512 # Distance between consecutive FFTs
EPS = 1e-8 # Epsilon value to avoid divide-by-zero errors

EFFNET_CHECKPOINT = "/kaggle/input/models/manoranjankumar7/effnet-b0/pytorch/default/9/effnetb0_final.ckpt"
RESNET_CHECKPOINT = "/kaggle/input/models/manoranjankumar7/resnet18-test/pytorch/default/10/resnet_final.ckpt"
SCRATCH_CHECKPOINT = "/kaggle/input/models/manoranjankumar7/scratchcnn/pytorch/default/2/scratchcnn_final.ckpt"

TRAIN_STATS = "/kaggle/input/models/manoranjankumar7/effnet-b0/pytorch/default/9/mel_stats_new.npz"
TEST_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
idx_to_genre = {i: g for i, g in enumerate(GENRES)}

BATCH_SIZE = 32

# 2. Helper functions

In [ ]:
def load_audio(path: str, sr=SR):
    """Load an audio file at the specified sampling rate and return the waveform."""
    y, s = librosa.load(path, sr=sr) # y = 1d numpy floating point array, s = sampling rate
    return y.astype(np.float32)

def pad_or_trim(y: np.ndarray, sr=SR, target_duration=TARGET_DURATION):
    """Return a fixed-duration waveform.
    
    If longer than target duration, trim start/end and return the middle.
    
    If shorter, right-pad with zeros.
    """
    target_length = int(sr * target_duration) # Hz * s
    if len(y) >= target_length:
        start = (len(y) - target_length) // 2
        return y[start: start+target_length] # Take the middle bit
    else:
        return np.pad(y, (0, target_length - len(y)), mode="constant", constant_values=0) # Right-pad with 0s

def log_mel(y: np.ndarray, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS):
    """
    Return a log-mel spectrogram for a waveform.

    The amplitude is scaled relative to `ref=1.0`.
    """
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels) # Power-scale mel spectrogram
    # Convert to dB for a log-mel spectrogram
    return librosa.power_to_db(S=S, ref=1.0, amin=1e-10) # ref=1, *not* np.max

# 3. Model definitions

The final three models I'm going with are: ResNet18 with a new classifier head, EfficientNet_B0 with a new classifier head, and a simple custom CNN I defined called ScratchCNN.

In [ ]:
class ResNet(nn.Module):
    """
    Input: (B, 1, n_mels, T)
    
    Output: logits (B, num_classes)
    """
    def __init__(self, num_classes=10, pretrained=True, freeze_backbone=False):
        super().__init__()
        res = models.resnet18(weights="DEFAULT" if pretrained else None)
        old_conv = res.conv1
        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )

        # Taking the mean of the old 3-channel input weights and converting it to 1 channel
        with torch.inference_mode():
            new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
        res.conv1 = new_conv
        self.backbone = res

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
            for m in self.backbone.modules(): # Set batchnorm layers to eval if freezing!
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eval()

        feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity() # Remove final FC layer from backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x: torch.Tensor, return_features=False) -> torch.Tensor:
        features = self.backbone(x) # (B, feat_dim)
        if return_features:
            return features
        logits = self.classifier(features) # (B, num_classes)
        return logits

In [ ]:
class EffNetB0(nn.Module):
    """
    Input: (B, 1, n_mels, T)
    
    Output: logits (B, num_classes)
    """
    def __init__(self, num_classes=10, pretrained=True, freeze_backbone=False):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights="DEFAULT" if pretrained else None)
        first_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=first_conv.out_channels,
            kernel_size=first_conv.kernel_size,
            stride=first_conv.stride,
            padding=first_conv.padding,
            bias=(first_conv.bias is not None)
        )

        with torch.inference_mode():
            old_w = first_conv.weight  # (out_ch, 3, k, k)
            new_conv.weight[:] = old_w.mean(dim=1, keepdim=True)
            if first_conv.bias is not None:
                new_conv.bias[:] = first_conv.bias
        self.backbone.features[0][0] = new_conv

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
            for m in self.backbone.modules(): # Set batchnorm layers to eval if freezing!
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eval()

        last_linear = self.backbone.classifier[-1]
        feat_dim = last_linear.in_features
            
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x: torch.Tensor, return_features=False) -> torch.Tensor:
        # x: (B, 1, H, W)
        features = self.backbone(x) # (B, feat_dim)
        if return_features:
            return features
        logits = self.classifier(features) # (B, num_classes)
        return logits

In [ ]:
# For defining the scratch CNN
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2,2))
        )
    def forward(self, x):
        return self.net(x)

class ScratchCNN(nn.Module):
    """
    Input: (B, 1, n_mels, T)

    Output: logits (B, num_classes)
    """
    def __init__(self, num_classes=10, in_ch=1, base_ch=32, dropout=0.4):
        super().__init__()
        # base_ch: 32 -> 64 -> 128 -> 256
        self.block1 = ConvBlock(in_ch, base_ch)
        self.block2 = ConvBlock(base_ch, base_ch*2)
        self.block3 = ConvBlock(base_ch*2, base_ch*4)
        # Extra conv layer to reduce spatial dimensions
        self.conv4 = nn.Sequential(
            nn.Conv2d(base_ch*4, base_ch*8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch*8),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1,1))   # (B, C, 1, 1)
        )

        feat_dim = base_ch * 8
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout*0.5),
            nn.Linear(512, num_classes)
        )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, 1, n_mels, T)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.conv4(x) # (B, C, 1, 1)
        x = x.view(x.size(0), -1) # (B, feat_dim)
        logits = self.classifier(x)
        return logits

# 4. Test dataset and ensemble prediction function

In [ ]:
class TestDataset(Dataset):
    def __init__(self, file_list, mean, std):
        self.files = [str(p) for p in file_list]
        # Reshape to (1, n_mels, 1) so broadcasting with (B, 1, n_mels, T) works
        self.mean = torch.tensor(mean, dtype=torch.float32).view(1, -1, 1)
        self.std  = torch.tensor(std, dtype=torch.float32).view(1, -1, 1)

    def __len__(self):
        return len(self.files)

    def get_mel(self, idx):
        return log_mel(pad_or_trim(load_audio(self.files[idx]))) # Don't peak-normalize y here

    def __getitem__(self, idx):
        mel = self.get_mel(idx) # (n_mels, time)
        mel_t = torch.from_numpy(mel).unsqueeze(0).float() # (1, n_mels, time)
        mel_t = (mel_t - self.mean) / self.std # Normalize (torch broadcasting: (1, n_mels, time) - (1, n_mels, 1))
        file_id = os.path.basename(self.files[idx])
        return {
            "id": file_id,
            "path": self.files[idx],
            "mel": mel_t
        }

I'll be using an ensemble of the three models to perform inference on the test set, as experimenting shows it yields the best performance.

In [ ]:
def ensemble_predictions(models, dataset: Dataset, device=None, batch_size=BATCH_SIZE, num_workers=0, weights=None):
    """Make predictions by ensembling model predictions on the test set."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    models = list(models)
    n_models = len(models)

    # Model ensemble weights
    if weights is None:
        weights = np.ones(n_models, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / (weights.sum() + EPS)

    for m in models:
        m.to(device)
        m.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    results = {}
    i = 1 # Count progress

    with torch.inference_mode():
        for batch in loader:
            print(i * batch_size)
            i += 1
            ids = batch["id"]
            x = batch["mel"].to(device, non_blocking=True)  # (B, 1, C, T)

            agg = None
            for w, m in zip(weights, models):
                logits = m(x) # (B, n_classes)
                logits_cpu = logits.detach().cpu().numpy()
                if agg is None:
                    agg = w * logits_cpu
                else:
                    agg += w * logits_cpu
            exp = np.exp(agg - np.max(agg, axis=1, keepdims=True))
            probs_batch = exp / (exp.sum(axis=1, keepdims=True) + EPS)

            for fid, p in zip(ids, probs_batch):
                results[fid] = p.copy()

    return results

# 5. Model initialization and loading checkpoints

In [ ]:
resnet_finetuned = ResNet(num_classes=10, pretrained=False)
ckpt = torch.load(RESNET_CHECKPOINT, map_location="cpu", weights_only=False)
resnet_finetuned.load_state_dict(ckpt["model_state"])

In [ ]:
effnet_finetuned = EffNetB0(num_classes=10, pretrained=False)
ckpt = torch.load(EFFNET_CHECKPOINT, map_location="cpu", weights_only=False)
effnet_finetuned.load_state_dict(ckpt["model_state"])

In [ ]:
scratch_cnn = ScratchCNN(num_classes=10, in_ch=1, base_ch=32)
ckpt = torch.load(SCRATCH_CHECKPOINT, map_location="cpu", weights_only=False)
scratch_cnn.load_state_dict(ckpt["model_state"])

# 6. Inference

Fingers crossed :)

In [ ]:
# Load the training set stats
stats = np.load(TRAIN_STATS, mmap_mode="r")
mean, std = stats["mean"], stats["std"]

# Create the TestDataset instance
test_files = sorted([f for f in Path(TEST_DIR).iterdir() if f.is_file()])
test_set = TestDataset(test_files, mean=mean, std=std)

# Load the models
models = [effnet_finetuned, resnet_finetuned, scratch_cnn]

# Inference (no ensemble weights)
preds = ensemble_predictions(models=models, dataset=test_set, num_workers=4)
rows = []
for id, probs, in preds.items():
    genre = idx_to_genre[np.argmax(probs)]
    rows.append((id, genre))
df = pd.DataFrame(rows, columns=["id", "genre"])
df["id"] = df["id"].str.extract(r"([0-9]+)") # Could've probably just used test.csv, but forgot about it until later >_<
df.to_csv("submission.csv", index=False)

!head submission.csv